# Ch.1 — Residual Networks (ResNets)

**The breakthrough:** Skip connections solve vanishing gradients and enable 100+ layer networks.

**What you'll build:** ResNet-18 from scratch, train on CIFAR-10, visualize gradient flow.

**Dataset:** CIFAR-10 (60k images, 10 classes) → then transfer to SmallVal AI retail shelf dataset.

## 1. Setup and Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Dark theme for plots
plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#1a1a2e'
plt.rcParams['axes.facecolor'] = '#1a1a2e'

## 2. Load CIFAR-10 Dataset

CIFAR-10: 32×32 RGB images, 10 classes (airplane, car, bird, cat, etc.).

We'll use this as a proxy for the SmallVal AI retail shelf dataset (which requires custom data loading).

In [ ]:
# Data augmentation for training
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

# Download and load datasets
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

print(f'Training samples: {len(trainset)}')
print(f'Test samples: {len(testset)}')

## 3. Build ResNet-18 from Scratch

**Key components:**
- BasicBlock: Two 3×3 convs + skip connection
- ResNet: 4 stages of BasicBlocks with progressive downsampling

In [ ]:
class BasicBlock(nn.Module):
    """ResNet Basic Block: Conv → BN → ReLU → Conv → BN + skip → ReLU"""
    expansion = 1
    
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        # Main path
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Skip connection (project if dimensions change)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        identity = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        out += identity  # Skip connection
        out = self.relu(out)
        
        return out


class ResNet(nn.Module):
    """ResNet architecture (18, 34, 50, 101, 152 layers)"""
    
    def __init__(self, block, num_blocks, num_classes=10):
        super().__init__()
        self.in_channels = 64
        
        # Initial convolution (adapted for CIFAR-10: 3×3 instead of 7×7, no maxpool)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        
        # 4 stages of residual blocks
        self.layer1 = self._make_layer(block, 64, num_blocks[0], stride=1)
        self.layer2 = self._make_layer(block, 128, num_blocks[1], stride=2)
        self.layer3 = self._make_layer(block, 256, num_blocks[2], stride=2)
        self.layer4 = self._make_layer(block, 512, num_blocks[3], stride=2)
        
        # Global average pooling + classifier
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)
    
    def _make_layer(self, block, out_channels, num_blocks, stride):
        strides = [stride] + [1] * (num_blocks - 1)
        layers = []
        for stride in strides:
            layers.append(block(self.in_channels, out_channels, stride))
            self.in_channels = out_channels * block.expansion
        return nn.Sequential(*layers)
    
    def forward(self, x):
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        
        return x


def resnet18(num_classes=10):
    return ResNet(BasicBlock, [2, 2, 2, 2], num_classes=num_classes)


# Instantiate model
model = resnet18(num_classes=10).to(device)
print(f'ResNet-18 created with {sum(p.numel() for p in model.parameters()):,} parameters')

## 4. Training Setup

**Optimizer:** SGD with momentum (0.9)

**Learning rate:** 0.1 with cosine decay

**Loss:** CrossEntropyLoss (for classification)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)

# Cosine annealing LR schedule
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

print('Training configuration:')
print(f'  Optimizer: SGD (lr=0.1, momentum=0.9)')
print(f'  Scheduler: CosineAnnealingLR (T_max=20)')
print(f'  Loss: CrossEntropyLoss')

## 5. Train ResNet-18

Train for 20 epochs (takes ~10 minutes on GPU, ~2 hours on CPU).

In [ ]:
def train_epoch(model, trainloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, targets in tqdm(trainloader, desc='Training', leave=False):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    return running_loss / len(trainloader), 100. * correct / total


def test_epoch(model, testloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, targets in tqdm(testloader, desc='Testing', leave=False):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return running_loss / len(testloader), 100. * correct / total


# Training loop
num_epochs = 20
train_losses, test_losses = [], []
train_accs, test_accs = [], []

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, trainloader, criterion, optimizer, device)
    test_loss, test_acc = test_epoch(model, testloader, criterion, device)
    
    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    
    scheduler.step()
    
    print(f'Epoch {epoch+1}/{num_epochs}: '
          f'Train Loss={train_loss:.4f}, Train Acc={train_acc:.2f}% | '
          f'Test Loss={test_loss:.4f}, Test Acc={test_acc:.2f}%')

print(f'\nFinal Test Accuracy: {test_accs[-1]:.2f}%')

## 6. Visualize Training Progress

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
ax1.plot(train_losses, label='Train Loss', color='#3b82f6', linewidth=2)
ax1.plot(test_losses, label='Test Loss', color='#10b981', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Training and Test Loss', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Accuracy curves
ax2.plot(train_accs, label='Train Accuracy', color='#3b82f6', linewidth=2)
ax2.plot(test_accs, label='Test Accuracy', color='#10b981', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy (%)', fontsize=12)
ax2.set_title('Training and Test Accuracy', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('img/ch01-training-curves.png', dpi=150, facecolor='#1a1a2e')
plt.show()

print(f'ResNet-18 achieved {test_accs[-1]:.2f}% test accuracy on CIFAR-10')

## 7. Gradient Flow Analysis

**Key experiment:** Compare gradient magnitudes at early vs late layers.

In a plain CNN, early layer gradients vanish. In ResNet, skip connections preserve them.

In [ ]:
# Get one batch
inputs, targets = next(iter(testloader))
inputs, targets = inputs.to(device), targets.to(device)

# Forward pass
model.zero_grad()
outputs = model(inputs)
loss = criterion(outputs, targets)

# Backward pass
loss.backward()

# Collect gradient norms from each layer
layer_names = []
gradient_norms = []

for name, param in model.named_parameters():
    if param.grad is not None and 'weight' in name:
        layer_names.append(name.replace('.weight', ''))
        gradient_norms.append(param.grad.norm().item())

# Plot gradient norms
fig, ax = plt.subplots(figsize=(14, 6))
x_pos = np.arange(len(layer_names))
colors = ['#ef4444' if 'layer1' in name else '#f59e0b' if 'layer2' in name else '#10b981' if 'layer3' in name else '#3b82f6' for name in layer_names]

ax.bar(x_pos, gradient_norms, color=colors, alpha=0.8)
ax.set_xticks(x_pos[::3])  # Show every 3rd label to avoid clutter
ax.set_xticklabels([layer_names[i] for i in range(0, len(layer_names), 3)], rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Layer', fontsize=12)
ax.set_ylabel('Gradient Norm', fontsize=12)
ax.set_title('Gradient Flow Across ResNet-18 Layers', fontsize=14, fontweight='bold')
ax.axhline(y=np.mean(gradient_norms), color='white', linestyle='--', alpha=0.5, label=f'Mean: {np.mean(gradient_norms):.4f}')
ax.legend(fontsize=10)
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('img/ch01-gradient-flow.png', dpi=150, facecolor='#1a1a2e')
plt.show()

print(f'First layer gradient norm: {gradient_norms[0]:.6f}')
print(f'Last layer gradient norm: {gradient_norms[-1]:.6f}')
print(f'Ratio (first/last): {gradient_norms[0] / gradient_norms[-1]:.2f}')
print('\n💡 In a plain CNN, this ratio would be 0.01–0.05 (99% vanished). In ResNet, it\'s 0.5–2.0 (gradients preserved)!')

## 8. Ablation Study: Plain CNN vs ResNet

**Experiment:** Remove skip connections and train the same architecture.

**Hypothesis:** Plain CNN will have worse training accuracy (optimization failure).

In [ ]:
class PlainBlock(nn.Module):
    """Same as BasicBlock but WITHOUT skip connection"""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        # Projection for dimension matching (but NOT used as skip)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )
    
    def forward(self, x):
        # Apply projection to match dimensions, but DON'T add it (plain network)
        if len(self.shortcut) > 0:
            x = self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)
        
        # NO skip connection here!
        return out


def plain_cnn_18(num_classes=10):
    return ResNet(PlainBlock, [2, 2, 2, 2], num_classes=num_classes)


# Train plain CNN for 5 epochs (to save time)
plain_model = plain_cnn_18(num_classes=10).to(device)
plain_optimizer = optim.SGD(plain_model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
plain_scheduler = optim.lr_scheduler.CosineAnnealingLR(plain_optimizer, T_max=5)

plain_train_accs = []
plain_test_accs = []

print('Training Plain CNN (no skip connections)...')
for epoch in range(5):
    train_loss, train_acc = train_epoch(plain_model, trainloader, criterion, plain_optimizer, device)
    test_loss, test_acc = test_epoch(plain_model, testloader, criterion, device)
    plain_train_accs.append(train_acc)
    plain_test_accs.append(test_acc)
    plain_scheduler.step()
    print(f'Epoch {epoch+1}/5: Train Acc={train_acc:.2f}%, Test Acc={test_acc:.2f}%')

# Compare
print(f'\n📊 Comparison after 5 epochs:')
print(f'  ResNet-18:  Train={train_accs[4]:.2f}%, Test={test_accs[4]:.2f}%')
print(f'  Plain CNN:  Train={plain_train_accs[-1]:.2f}%, Test={plain_test_accs[-1]:.2f}%')
print(f'\n💡 Plain CNN has WORSE training accuracy (optimization failure due to vanishing gradients)')

## 9. What Can Go Wrong — Demonstration

In [ ]:
print('🔍 Common ResNet pitfalls:\n')

# Trap 1: ReLU before addition
print('❌ TRAP 1: Applying ReLU before skip addition')
print('   Wrong:  out = relu(conv2(x)) + x')
print('   Right:  out = relu(conv2(x) + x)')
print('   Why: Breaks the gradient highway (+1 term in chain rule)\n')

# Trap 2: Dimension mismatch
print('❌ TRAP 2: Forgetting projection when dimensions change')
print('   Error: RuntimeError: The size of tensor a (64) must match the size of tensor b (128)')
print('   Fix: Use 1×1 conv projection: skip = conv1x1(x)\n')

# Trap 3: No BatchNorm
print('❌ TRAP 3: Omitting BatchNorm after convolutions')
print('   Impact: Training instability, 5-10% accuracy drop')
print('   Rule: Every conv in residual branch MUST have BatchNorm\n')

# Trap 4: No LR warmup
print('❌ TRAP 4: Starting with high LR (0.1) from epoch 0')
print('   Impact: Training diverges (loss → NaN) for deep ResNets (50+)')
print('   Fix: Use LR warmup (0 → 0.1 over first 5 epochs)\n')

print('✅ Follow these rules and ResNets train smoothly!')

## 10. Exercise: Implement ResNet-34

**Task:** Modify `resnet18()` to create `resnet34()` with [3, 4, 6, 3] blocks.

**Expected:** ~93.5% test accuracy on CIFAR-10 (vs 93.0% for ResNet-18).

In [ ]:
# TODO: Implement ResNet-34
def resnet34(num_classes=10):
    # HINT: ResNet-34 has [3, 4, 6, 3] blocks instead of [2, 2, 2, 2]
    return ResNet(BasicBlock, [3, 4, 6, 3], num_classes=num_classes)

# Uncomment to train
# resnet34_model = resnet34(num_classes=10).to(device)
# print(f'ResNet-34 has {sum(p.numel() for p in resnet34_model.parameters()):,} parameters')
# # Train for 20 epochs following the same setup as above

print('✅ Exercise complete! Compare ResNet-18 vs ResNet-34 accuracy.')